# Pipeline 4 - Social Post Donation Value Predictor

Generated: 2026-04-07T11:53:52.838076Z


## 1) Problem Framing

**Business question:** Predict which social posts are likely to drive donation value and referrals.

**Who cares:** nonprofit leadership, program staff, and/or fundraising staff depending on the pipeline domain.

**Why it matters:** this model turns operational, donor, or outreach data into a decision-support signal for a resource-constrained safehouse nonprofit.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model is evaluated on unseen data and is used for deployment-oriented scoring. The explanatory or relationship model is included to identify which variables appear most connected to the target and to support business interpretation. We do not treat predictive accuracy as causal proof.

**Success metrics:** classification pipelines use accuracy, F1, and ROC AUC where appropriate. Regression/forecasting pipelines use MAE, RMSE, and R-squared. The notebook also compares against a simple baseline so the results can be interpreted honestly.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on the pipeline-specific code for this business problem.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from data_prep import (
    as_bool,
    compare_profiles,
    dataset_profile,
    export_predictions_json,
    fill_numeric_median,
    get_project_paths,
    load_df,
    numeric,
    prep,
    quick_eda,
    score_bands,
    time_split,
)
from modeling import (
    CandidateResult,
    cv_evaluate_candidate_regression_transformed,
    cv_results_table,
    evaluate_candidate_regression_transformed,
    read_json_if_exists,
    select_simplest_within_delta_cv,
    should_export_by_guardrail,
    top_features,
    tune_model_randomized,
    within_tolerance_rate,
    write_ablation_report_json,
    write_run_report_json,
    ablate_feature_groups_one_by_one_cv_regression_transformed,
)

paths = get_project_paths()
print("Data dir:", paths.data_dir)
print("Output dir:", paths.out_dir)


def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)


## 2) Data Acquisition, Preparation, and Exploration

The pipeline reads the provided CSV files from `data/raw/`, performs joins and feature engineering in code, handles missing values reproducibly, and prints an EDA summary before modeling. The EDA step checks row counts, missingness, target distributions, feature summaries, and key categorical distributions.


In [ ]:
posts = load_df("social_media_posts")
posts["created_at"] = pd.to_datetime(posts["created_at"], errors="coerce")
posts = posts.dropna(subset=["created_at"]).sort_values("created_at").copy()
target = "estimated_donation_value_php"
num_all = ["post_hour","num_hashtags","mentions_count","caption_length","boost_budget_php","impressions","reach","likes","comments","shares","saves","click_throughs","video_views","engagement_rate","profile_visits","donation_referrals","follower_count_at_post","forwards",target]
for col in num_all:
    posts[col] = numeric(posts[col])
posts["has_call_to_action_bool"] = as_bool(posts["has_call_to_action"])
posts["features_resident_story_bool"] = as_bool(posts["features_resident_story"])
posts["is_boosted_bool"] = as_bool(posts["is_boosted"])
pre_features = ["platform","day_of_week","post_hour","post_type","media_type","num_hashtags","mentions_count","has_call_to_action_bool","call_to_action_type","content_topic","sentiment_tone","caption_length","features_resident_story_bool","campaign_name","is_boosted_bool","boost_budget_php"]
exp_features = pre_features + ["impressions","reach","likes","comments","shares","saves","click_throughs","video_views","engagement_rate","profile_visits","follower_count_at_post","forwards"]
cat_cols_pre = ["platform","day_of_week","post_type","media_type","call_to_action_type","content_topic","sentiment_tone","campaign_name"]
num_cols_pre = [c for c in pre_features if c not in cat_cols_pre]
cat_cols_exp = cat_cols_pre
num_cols_exp = [c for c in exp_features if c not in cat_cols_exp]
posts[cat_cols_pre] = posts[cat_cols_pre].fillna("Unknown")
fill_numeric_median(posts, list(set(num_cols_pre + num_cols_exp + [target])))
train, test = time_split(posts, "created_at")
print("Rows:", len(posts), "Train:", len(train), "Test:", len(test))
quick_eda(posts, "Social post donation-value modeling table", target_col=target, numeric_cols=num_cols_pre + [target], categorical_cols=cat_cols_pre)


## 3) Modeling and Feature Selection

The feature set is selected from fields that would be available at the decision point whenever possible. Predictive models use ensembles or tuned tree-based models when they improve out-of-sample performance. Explanatory models use simpler linear or logistic models when interpretability matters.


In [ ]:
# Planning model: uses only pre-post features available at creation time.
# Trains on log1p(target) but is evaluated/exported on the original PHP scale.
planning_candidates = {
    "LinearRegression": Pipeline([("pre", prep(cat_cols_pre, num_cols_pre)), ("model", LinearRegression())]),
    "GradientBoosting": Pipeline([("pre", prep(cat_cols_pre, num_cols_pre)), ("model", GradientBoostingRegressor(random_state=42))]),
    "RandomForestSmall": Pipeline([("pre", prep(cat_cols_pre, num_cols_pre)), ("model", RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=5))]),
}

# Relationship / learning model: allowed to use engagement metrics (post-hoc analysis only).
engagement_model = Pipeline([("pre", prep(cat_cols_exp, num_cols_exp)), ("model", GradientBoostingRegressor(random_state=42))])



## 4) Evaluation and Selection

The notebook uses a train/test or time-based holdout split depending on the business problem. Where appropriate, it also uses compact cross-validation, model comparison tables, and lightweight randomized tuning. Metrics are interpreted in business terms rather than treated as abstract statistics.


In [ ]:
# Strict holdout discipline: CV on train, single touch on time-ordered holdout.
X_train, y_train = train[pre_features], train[target]
X_holdout, y_holdout = test[pre_features], test[target]

report_path = paths.reports_dir / "social-post-donation-referrals_run_report.json"
prev_report = read_json_if_exists(report_path)

current_profile = dataset_profile(train[pre_features], categorical_cols=cat_cols_pre, numeric_cols=num_cols_pre)
drift = compare_profiles(current_profile, (prev_report or {}).get("data_profile"))

baseline_value = float(np.median(y_train))
print("Baseline (median):", {"baseline_value": baseline_value, "mae": float(np.mean(np.abs(y_holdout - baseline_value)))})

# 1) CV tournament (planning candidates) evaluated on PHP scale
cv_results = []
for name, est in planning_candidates.items():
    cv_results.append(
        cv_evaluate_candidate_regression_transformed(
            name,
            est,
            X_train,
            y_train,
            transform_y=np.log1p,
            inverse_y=np.expm1,
            clip_nonnegative=True,
        )
    )
cv_tbl = cv_results_table(cv_results).sort_values(by=["mae_mean", "fit_s_mean"], ascending=True)
print("Planning CV tournament (train only, PHP scale):")
print(cv_tbl.to_string(index=False))

best_mae = float(cv_tbl["mae_mean"].min())
delta = 0.05 * best_mae
selected_cv = select_simplest_within_delta_cv(cv_results, primary_metric="mae", delta=delta, higher_is_better=False)
base_selected = selected_cv.estimator
base_selected_name = selected_cv.name

# 2) Tune selected family (on transformed target)
param_space = {}
if base_selected_name.startswith("RandomForest"):
    param_space = {
        "model__n_estimators": [100, 150, 200, 300],
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_leaf": [1, 3, 5, 8],
    }
elif base_selected_name.startswith("GradientBoosting"):
    param_space = {
        "model__n_estimators": [100, 150, 250],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
    }

if param_space:
    tune = tune_model_randomized(
        base_selected_name,
        base_selected,
        X_train,
        np.log1p(y_train),
        param_distributions=param_space,
        task="regression",
        cv_metric="neg_mean_absolute_error",
        n_iter=10,
    )
    tuned_estimator = tune.best_estimator
    tuned_name = f"{base_selected_name}+tuned"
    tuned_params = tune.best_params
else:
    tuned_estimator = base_selected
    tuned_name = base_selected_name
    tuned_params = {}

# 3) Post-tune CV check on PHP scale
post_cv = cv_evaluate_candidate_regression_transformed(
    tuned_name,
    tuned_estimator,
    X_train,
    y_train,
    transform_y=np.log1p,
    inverse_y=np.expm1,
    clip_nonnegative=True,
)

# 4) Ablation (CV) on PHP scale
feature_groups = [[c] for c in pre_features]
ablation_tbl = ablate_feature_groups_one_by_one_cv_regression_transformed(
    base_name=tuned_name,
    estimator=tuned_estimator,
    X=X_train,
    y=y_train,
    transform_y=np.log1p,
    inverse_y=np.expm1,
    clip_nonnegative=True,
    feature_groups=feature_groups,
    primary_metric="mae",
    higher_is_better=False,
)
write_ablation_report_json(paths.reports_dir / "social-post-donation-referrals_ablation_report.json", {"pipeline": "social-post-donation-referrals", "table": ablation_tbl.to_dict(orient="records")})

# 5) Final holdout evaluation (single touch)
planning_model = tuned_estimator
planning_model_name = tuned_name
planning_model.fit(X_train, np.log1p(y_train))
holdout_pred = np.maximum(0, np.expm1(planning_model.predict(X_holdout)))
holdout_metrics = {
    "mae": float(np.mean(np.abs(y_holdout - holdout_pred))),
    "within_50pct": within_tolerance_rate(y_holdout, holdout_pred, tolerance=0.50, relative=True),
}

allow_export, guardrail = should_export_by_guardrail(
    previous_report=prev_report,
    current_holdout={"mae": holdout_metrics["mae"]},
    primary_metric="mae",
    min_delta_allowed=0.05 * float((prev_report or {}).get("holdout", {}).get("mae", holdout_metrics["mae"])),
    higher_is_better=False,
)

# Relationship / engagement model (post-hoc only) — evaluated on holdout for learning.
engagement_model.fit(train[exp_features], np.log1p(train[target]))
eng_pred = np.maximum(0, np.expm1(engagement_model.predict(test[exp_features])))
eng_metrics = {"mae": float(np.mean(np.abs(test[target] - eng_pred)))}

write_run_report_json(
    report_path,
    {
        "pipeline": "social-post-donation-referrals",
        "primary_metric": "mae",
        "selected_family": base_selected_name,
        "tuned_name": tuned_name,
        "tuned_params": tuned_params,
        "cv_table": cv_tbl.to_dict(orient="records"),
        "post_tune_cv": post_cv.metrics_mean,
        "holdout": holdout_metrics,
        "engagement_holdout": eng_metrics,
        "guardrail": guardrail,
        "data_profile": current_profile,
        "drift": drift,
        "notes": {
            "split": "time_split(created_at)",
            "planning_features": pre_features,
            "engagement_features_allowed_posthoc": [c for c in exp_features if c not in pre_features],
        },
    },
)
print("Wrote run report:", report_path)

if not allow_export:
    print("Export blocked by guardrail:", guardrail)



## 5) Causal and Relationship Analysis

The relationship analysis section highlights important features and discusses whether those relationships make sense for the organization. These findings are observational: they can guide hypotheses and strategy, but they are not automatically causal. Any sensitive resident-care decision must remain human-reviewed.


In [ ]:
print("Top planning model features:")
print(top_features(planning_model).to_string(index=False))
print("Top engagement relationship features:")
print(top_features(engagement_model).to_string(index=False))
print_business_takeaway(
    "For planning, use pre-post signals such as platform, format, CTA, and boost budget. "
    "For retrospective learning, engagement metrics explain more donation-value variance."
)


## 6) Deployment Notes

The final scoring step exports JSON to `output/ml-predictions/`. These files match the API import contract used by `POST /api/ml/import?replace=true` and can be viewed in the deployed staff portal under `/app/ml` or the ML action center.


In [ ]:
posts_out = posts[["post_id"] + pre_features].copy()
posts_out["predicted_value_php"] = np.maximum(0, np.expm1(planning_model.predict(posts_out[pre_features])))
posts_out["value_band"] = score_bands(posts_out["predicted_value_php"])

if allow_export:
    export_predictions_json(
        "post_donation_value",
        "SocialPost",
        posts_out[[
            "post_id",
            "predicted_value_php",
            "value_band",
            "platform",
            "post_type",
            "media_type",
            "post_hour",
            "has_call_to_action_bool",
            "is_boosted_bool",
            "boost_budget_php",
        ]].assign(export_model=planning_model_name),
        "post_id",
        "predicted_value_php",
        "value_band",
    )
else:
    print("Skipping export due to guardrail.")

